# External-Dataset Evaluation

Scores your already-trained, frozen checkpoints (flat classifier, shared
three-task model, dedicated Task1/Task2 models) against **any external image
cohort** you point it at — no training happens in this notebook.

## Why this notebook exists instead of running the repo's HIBA scripts directly

This project's designated mandatory external-evaluation dataset is **HIBA**
(`configs/evaluation/phase10_hiba_frozen_zero_shot.yaml`, ISIC collection 251,
DOI `10.34970/587329`). That config is currently locked:

```yaml
protocol_status: protocol_pending_dataset_audit
execution:
  dataset_audit_only_authorized_now: true
  inference_authorized: false
```

and `configs/datasets/hiba_external_label_mapping.yaml` itself has unresolved
entries ("pending official metadata inventory and human review") — the benign
diagnosis vocabulary isn't mapped yet. No `data/manifests/hiba_dataset_manifest.csv`
exists. Finishing that acquisition/audit (`scripts/acquire_hiba_official_metadata.py`,
`scripts/audit_hiba_external_dataset.py`) and approving the label mapping is a
human decision this notebook does not make for you.

**What this notebook does instead**: implements the same *decision policy*
the HIBA config already commits to for any external cohort —

```yaml
decision_policy:
  flat: argmax
  hierarchy_stage_1: argmax
  hierarchy_stage_2: argmax
  external_calibration_fitting_allowed: false
  external_threshold_fitting_allowed: false
  checkpoint_selection_allowed: false
```

— i.e. frozen checkpoints only, plain `argmax` decisions, **no** threshold or
calibration tuning against the external data, **no** re-selecting a different
checkpoint based on how well it does here. You can point this at HIBA once
its audit and label mapping are finalized (by building the CSV described
below from the approved HIBA manifest), or at any other external cohort in
the meantime.

## What you need to prepare

One CSV file with these columns:

| column | required | meaning |
|---|---|---|
| `image_path` | yes | absolute path, or a path relative to `EXTERNAL_IMAGES_ROOT` (set below) |
| `label` | yes | one of `non_malignant`, `melanoma`, `bcc`, `scc` — already mapped to this project's four endpoint classes |
| `image_id` | no | a stable identifier; falls back to the row index if absent |

Nothing else is required. This notebook does not care which real-world
dataset the rows came from, how they were licensed, or how the label mapping
was derived — that governance step (exactly what the HIBA audit scripts
above exist to do) is on you to complete before trusting the label column.


## 0. Setup

Same Windows/pickling compatibility fix as every other notebook in this
project — teaches Python's `pickle` module to serialize `MappingProxyType`,
which this repo's dataset classes use and which Windows DataLoader worker
processes need to pickle.

In [ ]:
import copyreg
import platform
import sys
from pathlib import Path
from types import MappingProxyType

import torch


def _rebuild_mappingproxy(mapping):
    return MappingProxyType(mapping)


def _reduce_mappingproxy(obj):
    return _rebuild_mappingproxy, (dict(obj),)


copyreg.pickle(MappingProxyType, _reduce_mappingproxy)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"project_root={PROJECT_ROOT}")
print(f"device={DEVICE}")


## 1. Point this at your external dataset and your trained checkpoints

Fill in `EXTERNAL_DATASET_CSV_PATH` (the CSV described above) and, if your
`image_path` column holds paths relative to some folder rather than
absolute paths, set `EXTERNAL_IMAGES_ROOT` to that folder (otherwise leave
it as `None` and `image_path` values are used exactly as given, or resolved
relative to `PROJECT_ROOT`).

Reuse the exact checkpoint paths from notebook 08 (or your own later runs).


In [ ]:
EXTERNAL_DATASET_NAME = "REPLACE_WITH_A_SHORT_NAME"  # e.g. "hiba" -- used only for naming output files
EXTERNAL_DATASET_CSV_PATH = PROJECT_ROOT / "REPLACE_WITH_YOUR_EXTERNAL_DATASET.csv"
EXTERNAL_IMAGES_ROOT = None  # e.g. PROJECT_ROOT / "data/raw/hiba/images" -- or leave None

SHARED_CHECKPOINT_PATH = PROJECT_ROOT / "runs/phase03_shared_three_task/seed_42_local/best_checkpoint.pt"
TASK1_STANDALONE_CHECKPOINT_PATH = PROJECT_ROOT / "REPLACE_WITH_YOUR_TASK1_RUN/best_checkpoint.pt"
TASK2_STANDALONE_CHECKPOINT_PATH = PROJECT_ROOT / "REPLACE_WITH_YOUR_TASK2_RUN/best_checkpoint.pt"

# Optional: the flat classifier from notebook 01.
FLAT_CHECKPOINT_PATH = None  # e.g. PROJECT_ROOT / "experiments/runs/full__phase06_.../best_checkpoint.pt"

for name, path in {
    "external_dataset_csv": EXTERNAL_DATASET_CSV_PATH,
    "shared": SHARED_CHECKPOINT_PATH,
    "task1_standalone": TASK1_STANDALONE_CHECKPOINT_PATH,
    "task2_standalone": TASK2_STANDALONE_CHECKPOINT_PATH,
}.items():
    print(f"{name}: {path} exists={Path(path).is_file()}")


## 2. Preprocessing

A small, generic dataset class that reads your CSV and applies this repo's
locked, deterministic evaluation transform (`build_eval_transform` —
resize-256, center-crop-224, ImageNet normalization; the exact same one used
for every validation/internal-test split elsewhere in this project, with no
augmentation, so external scoring is directly comparable to internal-test
numbers). This class is new (a generic CSV-driven dataset does not exist in
`src/`, since every existing dataset class there is manifest-schema-specific
to ISIC2019 or the EMB Stage-3 source) — everything it depends on
(`build_eval_transform`, image class labels) is unchanged repo code.


In [ ]:
import pandas as pd
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from src.data.transforms import build_eval_transform

FINAL_CLASS_NAMES = ["non_malignant", "melanoma", "bcc", "scc"]
FINAL_CLASS_TO_INDEX = {name: index for index, name in enumerate(FINAL_CLASS_NAMES)}


class ExternalCohortDataset(Dataset):
    """Minimal (image_path, label) dataset for any external evaluation cohort."""

    def __init__(self, csv_path, images_root, transform):
        frame = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
        required = {"image_path", "label"}
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f"External dataset CSV is missing columns: {sorted(missing)}")

        unknown_labels = sorted(set(frame["label"]) - set(FINAL_CLASS_TO_INDEX))
        if unknown_labels:
            raise ValueError(
                f"Unknown label values in external dataset CSV: {unknown_labels}; "
                f"expected one of {FINAL_CLASS_NAMES}."
            )

        self.transform = transform
        self.images_root = Path(images_root) if images_root is not None else None
        self.image_paths = frame["image_path"].tolist()
        self.labels = frame["label"].tolist()
        self.image_ids = (
            frame["image_id"].tolist()
            if "image_id" in frame.columns
            else [str(i) for i in range(len(frame))]
        )

    def _resolve_path(self, raw_path):
        candidate = Path(raw_path)
        if candidate.is_absolute():
            return candidate
        if self.images_root is not None:
            return self.images_root / candidate
        return PROJECT_ROOT / candidate

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        path = self._resolve_path(self.image_paths[index])
        with Image.open(path) as opened:
            image = opened.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        target = FINAL_CLASS_TO_INDEX[self.labels[index]]
        return {
            "image": image,
            "target": torch.tensor(target, dtype=torch.long),
            "image_id": self.image_ids[index],
        }


eval_transform = build_eval_transform()
external_dataset = ExternalCohortDataset(EXTERNAL_DATASET_CSV_PATH, EXTERNAL_IMAGES_ROOT, eval_transform)
external_loader = DataLoader(
    external_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,  # kept at 0 for the same local-Windows-DataLoader reasons as the other notebooks
    pin_memory=(DEVICE == "cuda"),
    drop_last=False,
)

print(f"external dataset: {len(external_dataset)} images")
label_counts = pd.Series(external_dataset.labels).value_counts()
print(label_counts)


## 3. Model definition

Identical model-building code to notebook 08 — same architectures, same
checkpoint-loading pattern. `pretrained="none"` only skips a redundant
ImageNet download since every weight gets overwritten by the checkpoint.


In [ ]:
from src.models.classification_backbone import build_classification_model
from src.models.shared_three_task import build_shared_three_task_model

shared_payload = torch.load(SHARED_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
shared_architecture = shared_payload["model_metadata"]["architecture"]
shared_model = build_shared_three_task_model(shared_architecture, pretrained="none")
shared_model.load_state_dict(shared_payload["model_state_dict"], strict=True)
shared_model.to(DEVICE).eval()

task1_payload = torch.load(TASK1_STANDALONE_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
task1_config = task1_payload["config"]
task1_model = build_classification_model(
    task1_config["model"]["architecture"],
    task1_config["model"]["number_of_classes"],
    pretrained="none",
    dropout_probability=task1_config["model"].get("dropout_probability", 0.2),
)
task1_model.load_state_dict(task1_payload["model_state_dict"], strict=True)
task1_model.to(DEVICE).eval()
assert list(task1_payload["class_names"]) == ["non_malignant", "malignant"]

task2_payload = torch.load(TASK2_STANDALONE_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
task2_config = task2_payload["config"]
task2_model = build_classification_model(
    task2_config["model"]["architecture"],
    task2_config["model"]["number_of_classes"],
    pretrained="none",
    dropout_probability=task2_config["model"].get("dropout_probability", 0.2),
)
task2_model.load_state_dict(task2_payload["model_state_dict"], strict=True)
task2_model.to(DEVICE).eval()
assert list(task2_payload["class_names"]) == ["melanoma", "bcc", "scc"]

flat_model = None
if FLAT_CHECKPOINT_PATH is not None:
    flat_payload = torch.load(FLAT_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    flat_config = flat_payload["config"]
    flat_model = build_classification_model(
        flat_config["model"]["architecture"],
        flat_config["model"]["number_of_classes"],
        pretrained="none",
        dropout_probability=flat_config["model"].get("dropout_probability", 0.2),
    )
    flat_model.load_state_dict(flat_payload["model_state_dict"], strict=True)
    flat_model.to(DEVICE).eval()
    assert list(flat_payload["class_names"]) == FINAL_CLASS_NAMES

print("models loaded")


## 4. Run every model once over the external cohort

One forward pass per model, collecting raw probabilities. Unmasked, so the
same collected probabilities feed both the hard-routing and soft-routing
compositions below without re-running inference.


In [ ]:
import numpy as np

def collect_external_probabilities(shared_model, task1_model, task2_model, flat_model, dataloader, device):
    image_ids, targets = [], []
    shared_p1, shared_p2 = [], []
    dedicated_p1, dedicated_p2 = [], []
    flat_probs = [] if flat_model is not None else None

    with torch.inference_mode():
        for batch in dataloader:
            images = batch["image"].to(device, non_blocking=True)

            shared_out = shared_model(images)
            shared_p1.append(torch.softmax(shared_out["task1"].float(), dim=1).cpu())
            shared_p2.append(torch.softmax(shared_out["task2"].float(), dim=1).cpu())

            dedicated_p1.append(torch.softmax(task1_model(images).float(), dim=1).cpu())
            dedicated_p2.append(torch.softmax(task2_model(images).float(), dim=1).cpu())

            if flat_model is not None:
                flat_probs.append(torch.softmax(flat_model(images).float(), dim=1).cpu())

            image_ids.extend(batch["image_id"])
            targets.append(batch["target"])

    result = {
        "image_ids": image_ids,
        "targets": torch.cat(targets).numpy(),
        "shared_p1": torch.cat(shared_p1).numpy(),
        "shared_p2": torch.cat(shared_p2).numpy(),
        "dedicated_p1": torch.cat(dedicated_p1).numpy(),
        "dedicated_p2": torch.cat(dedicated_p2).numpy(),
    }
    if flat_probs is not None:
        result["flat_probs"] = torch.cat(flat_probs).numpy()
    return result


external = collect_external_probabilities(shared_model, task1_model, task2_model, flat_model, external_loader, DEVICE)
external_targets = external["targets"]
external_stage1_targets = (external_targets != 0).astype(np.int64)
external_stage2_targets = np.where(external_targets == 0, -1, external_targets - 1)

print(f"collected probabilities for {len(external['image_ids'])} external images")
print(f"true malignant count: {int(external_stage1_targets.sum())} / {len(external_stage1_targets)}")


## 5. Decisions: soft routing (this project's proposed fix) and hard routing (the paper's existing baseline)

Both are computed from the same collected probabilities, following the
project's external decision policy: plain `argmax`, no threshold or
calibration fitting against this external cohort.

`compose_soft_routing` is the same chain-rule composition from notebook 08.
`build_hierarchical_routing` (`src/evaluation/hierarchical_evaluator.py`) is
the repo's own existing hard-routing function, reused unchanged — it needs
Stage-2 predictions set to `-1` wherever hard routing would never have
executed Task 2, which is why the masking below duplicates
`collect_shared_isic_predictions`'s masking rule, not the unmasked
soft-routing probabilities from the cell above.


In [ ]:
from src.evaluation.classification_metrics import compute_classification_metrics
from src.evaluation.hierarchical_evaluator import build_hierarchical_routing


def compose_soft_routing(p1, p2, *, stage1_targets=None, oracle=False):
    """Chain-rule composition of Task1 (2-class) and Task2 (3-class)
    probabilities into one 4-class distribution -- see notebook 08 for the
    full derivation. Not present in src/, since the repo only implements
    hard routing there.
    """
    if oracle:
        if stage1_targets is None:
            raise ValueError("oracle routing requires stage1_targets.")
        p_malignant = stage1_targets.astype(np.float64)
    else:
        p_malignant = p1[:, 1]
    p_non_malignant = 1.0 - p_malignant
    composed = np.stack(
        [p_non_malignant, p_malignant * p2[:, 0], p_malignant * p2[:, 1], p_malignant * p2[:, 2]],
        axis=1,
    )
    return composed / composed.sum(axis=1, keepdims=True)


def predicted_and_metrics(probabilities, targets):
    predictions = probabilities.argmax(axis=1)
    return predictions, compute_classification_metrics(targets, predictions, FINAL_CLASS_NAMES)


def hard_routing_metrics(p1, p2, stage1_targets, stage2_targets):
    """Reuse the repo's own build_hierarchical_routing exactly as
    collect_shared_isic_predictions (src/evaluation/phase04_comparative_harness.py)
    prepares its inputs: Stage-2 predictions masked to -1 outside the
    hard-routing execution set.
    """
    stage1_preds = p1.argmax(axis=1)
    stage2_preds_raw = p2.argmax(axis=1)
    execution = (stage1_targets == 1) | (stage1_preds == 1)
    stage2_preds = np.full_like(stage1_preds, -1)
    stage2_preds[execution] = stage2_preds_raw[execution]

    routing = build_hierarchical_routing(stage1_targets, stage1_preds, stage2_targets, stage2_preds)
    predicted_metrics = compute_classification_metrics(
        routing.final_targets, routing.predicted_gate_predictions, FINAL_CLASS_NAMES
    )
    oracle_metrics = compute_classification_metrics(
        routing.final_targets, routing.oracle_gate_predictions, FINAL_CLASS_NAMES
    )
    return predicted_metrics, oracle_metrics, routing.routing_counts


results = {}

# --- Shared model ---
shared_soft_pred_probs = compose_soft_routing(external["shared_p1"], external["shared_p2"])
_, results["shared_soft_predicted"] = predicted_and_metrics(shared_soft_pred_probs, external_targets)
shared_soft_oracle_probs = compose_soft_routing(
    external["shared_p1"], external["shared_p2"], stage1_targets=external_stage1_targets, oracle=True
)
_, results["shared_soft_oracle"] = predicted_and_metrics(shared_soft_oracle_probs, external_targets)
results["shared_hard_predicted"], results["shared_hard_oracle"], shared_routing_counts = hard_routing_metrics(
    external["shared_p1"], external["shared_p2"], external_stage1_targets, external_stage2_targets
)

# --- Dedicated (standalone) models ---
dedicated_soft_pred_probs = compose_soft_routing(external["dedicated_p1"], external["dedicated_p2"])
_, results["dedicated_soft_predicted"] = predicted_and_metrics(dedicated_soft_pred_probs, external_targets)
dedicated_soft_oracle_probs = compose_soft_routing(
    external["dedicated_p1"], external["dedicated_p2"], stage1_targets=external_stage1_targets, oracle=True
)
_, results["dedicated_soft_oracle"] = predicted_and_metrics(dedicated_soft_oracle_probs, external_targets)
results["dedicated_hard_predicted"], results["dedicated_hard_oracle"], dedicated_routing_counts = hard_routing_metrics(
    external["dedicated_p1"], external["dedicated_p2"], external_stage1_targets, external_stage2_targets
)

# --- Flat classifier ---
if "flat_probs" in external:
    _, results["flat"] = predicted_and_metrics(external["flat_probs"], external_targets)

for name, metrics in results.items():
    print(f"[{name}] macro_f1={metrics['macro_f1']:.4f} balanced_accuracy={metrics['balanced_accuracy']:.4f}")

print("\nshared routing counts:", shared_routing_counts)
print("dedicated routing counts:", dedicated_routing_counts)


## 6. Comparison table and chart

In [ ]:
import pandas as pd

comparison_table = pd.DataFrame(
    [{"system": name, **metrics} for name, metrics in results.items()]
)[["system", "macro_f1", "balanced_accuracy", "accuracy", "weighted_f1"]]
display(comparison_table)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(comparison_table["system"], comparison_table["macro_f1"], color="#1f77b4")
ax.set_ylabel("Macro-F1 (external cohort)")
ax.set_title(f"External-cohort evaluation: {EXTERNAL_DATASET_NAME}")
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 7. Save outputs

Writes to a dataset-named subdirectory under `experiments/evaluations/external_evaluation/`
so results from different external cohorts never collide or get overwritten.


In [ ]:
import json
from datetime import datetime, timezone

output_dir = PROJECT_ROOT / "experiments" / "evaluations" / "external_evaluation" / EXTERNAL_DATASET_NAME
output_dir.mkdir(parents=True, exist_ok=True)

comparison_table.to_csv(output_dir / "external_comparison_table.csv", index=False)

summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "external_dataset_name": EXTERNAL_DATASET_NAME,
    "external_dataset_csv_path": str(EXTERNAL_DATASET_CSV_PATH),
    "external_sample_count": len(external["image_ids"]),
    "external_label_distribution": pd.Series(external_dataset.labels).value_counts().to_dict(),
    "shared_checkpoint_path": str(SHARED_CHECKPOINT_PATH),
    "task1_standalone_checkpoint_path": str(TASK1_STANDALONE_CHECKPOINT_PATH),
    "task2_standalone_checkpoint_path": str(TASK2_STANDALONE_CHECKPOINT_PATH),
    "flat_checkpoint_path": str(FLAT_CHECKPOINT_PATH) if FLAT_CHECKPOINT_PATH else None,
    "shared_routing_counts": shared_routing_counts,
    "dedicated_routing_counts": dedicated_routing_counts,
    "results": results,
}
with open(output_dir / "external_evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, sort_keys=True)

print(f"wrote {output_dir / 'external_comparison_table.csv'}")
print(f"wrote {output_dir / 'external_evaluation_summary.json'}")


## 8. Interpreting external results for the paper

An external cohort is a much harder test than the internal-test split — it
was never seen during training, validation-based model selection, or the
design of the transforms/label mapping. Expect **all** systems to score
lower here than on internal test. What's worth reporting:

1. **Does soft routing's advantage (if any) hold up externally**, or was it
   specific to the internal-test distribution? Compare
   `shared_soft_predicted` vs. `shared_hard_predicted` here against the same
   comparison on internal test (notebook 08).
2. **Does the flat-vs-hierarchy ranking flip** on this external cohort
   compared to internal test? A ranking that survives external evaluation is
   much stronger evidence than one that only held on the internal split.
3. **Routing counts** (`shared_routing_counts`, `dedicated_routing_counts`)
   — do malignant-block and non-malignant-misroute rates get worse
   externally? A large jump suggests the routing gate itself doesn't
   generalize, independent of Task 2's subtype accuracy.

Remember this notebook's own honesty constraint: nothing here was fit,
thresholded, or calibrated against the external cohort — these are true
zero-shot numbers, consistent with what the paper's external-evaluation
section needs to claim.
